In [19]:
import matplotlib.pyplot as plt
import sys
from typing import List, Dict
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import numpy
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import itertools
import math
import tqdm
import libsbml
import pickle
import os
import sklearn

In [20]:
INPUTS = 5
OUTPUTS = 5

class PRRegressor_ffnn(nn.Module):
    def __init__(self, inputs, layers, outputs, activation):
        super().__init__()
        all_layers = [inputs] + layers + [outputs]
        self.linears = nn.ModuleList([nn.Linear(all_layers[i], all_layers[i + 1]) for i in range(len(all_layers) - 1)])
        if activation == "tanh":
            self.act = nn.Tanh()
        elif activation == "relu":
            self.act = nn.ReLU()
    
    def forward(self, x):
        for hidden in self.linears[:-1]:
            x = self.act(hidden(x))
        x = self.linears[-1](x)
        return x

nums_nodes_layers_s_ffnn = [[13, 13], [15, 15], [13, 13, 13]]

In [21]:
NUM_HIDDEN = 1

class PR_rnn(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, activation):
        super(PR_rnn, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity=activation)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out

nums_nodes_one_layer = [10, 15, 19]

In [22]:
num_nodes = {"ffnn": nums_nodes_layers_s_ffnn,
             "ernn": nums_nodes_one_layer}

In [23]:
fname = "/home/eka/thesis/ParkRam2/ParkRam_constant_feed.xlsx"
data = pandas.read_excel(fname, sheet_name=1)

In [24]:
def convert_symbol_to_label(symbol):
    return f"{symbol}_noise"

def convert_symbol_to_sd(symbol):
    return f"sd_{symbol}"

In [25]:
with open("../PR_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])

In [26]:
prev_val = 0
batches_lengths = []
for val in data["exp"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1

not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [27]:
maxes = dict()
mins = dict()
concentrations_symbols = ["X", "S", "PT", "PM", "V"]
true_symbols = ["X True", "S true", "PT true", "PM true", "V true"]
concentrations_labels = [convert_symbol_to_label(symbol) for symbol in concentrations_symbols]

for label in (['t'] + concentrations_labels):
    max_val = max(data[data["exp"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["exp"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)

mins_tensor = torch.Tensor([mins[label] for label in concentrations_labels])
maxes_tensor = torch.Tensor([maxes[label] for label in concentrations_labels])

#sd scaling
for symbol in concentrations_symbols:
    label_sd = convert_symbol_to_sd(symbol)
    label_sd_scaled = f"{label_sd}_scaled"
    label_concentration = convert_symbol_to_label(symbol)
    data[label_sd_scaled] = data[label_sd] / (maxes[label_concentration] - mins[label_concentration])

In [28]:
prev_val = 0
batches_lengths = []
for val in data["exp"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ["X_noise", "S_noise", "V_noise", "Feed"]
first_of_each_batch = data[changing_vars].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)



In [29]:
def get_train_val_test_data(train_val_test, batches):
    trainbatch, valbatch, testbatch = train_val_test
    Xs_train = []
    ys_train = []
    Xs_val = []
    ys_val = []
    Xs_test = []
    ys_test = []
    for i, batch in enumerate(batches):
        scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
        X = pandas.concat([batch[["t_scaled"]], scaled_initials_for_this_batch], axis=1)
        y = batch[concentrations_labels]
        X = torch.Tensor(X.values)
        y = torch.Tensor(y.values)
        if i in testbatch:
            Xs_test.append(X)
            ys_test.append(y)
        elif i in valbatch:
            Xs_val.append(X)
            ys_val.append(y)
        elif i in trainbatch:
            Xs_train.append(X)
            ys_train.append(y)
    Xs_train = torch.stack(Xs_train)
    Xs_test = torch.stack(Xs_test)
    Xs_val = torch.stack(Xs_val)
    ys_test_true = torch.flatten(torch.stack(ys_test))
    ys_train_true = torch.flatten(torch.stack(ys_train))
    ys_val_true = torch.flatten(torch.stack(ys_val))
    return Xs_train, Xs_val, Xs_test, ys_train_true, ys_val_true, ys_test_true

In [30]:
def wmse(input, target, sd):
    return torch.sum(((input - target) / sd) ** 2) / input.numel()

loss_fn = wmse

In [31]:
Xs_ys_sds = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["t_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    sd = batch[[f"{convert_symbol_to_sd(symbol)}_scaled" for symbol in concentrations_symbols]]
    X = torch.Tensor(X.values)
    y = torch.Tensor(y.values)
    sd = torch.Tensor(sd.values)
    Xs_ys_sds.append((X, y, sd))

In [32]:
def calc_loss(nn_model, nums):
    this_batches = [batch for i, batch in enumerate(Xs_ys_sds) if i in nums]
    X_train = torch.stack([batch[0] for batch in this_batches])
    y_train = torch.stack([batch[1] for batch in this_batches])
    sd_train = torch.stack([batch[2] for batch in this_batches])
    
    data = TensorDataset(X_train, y_train, sd_train)
    dataloader = DataLoader(data, len(data))
    sum_loss = 0
    num_loss = 0
    for X, y, sd in dataloader:
        y_pred = nn_model(X)
        loss = loss_fn(y_pred, y, sd)
        sum_loss += loss
        num_loss += 1
    return float(sum_loss / num_loss)

In [33]:
def load_model(path_to_folder, filename, model_type, activation, layers):
    if model_type == "ffnn":
        nn_model = PRRegressor_ffnn(INPUTS, layers, OUTPUTS, activation)
    elif model_type == "ernn":
        nn_model = PR_rnn(INPUTS, layers, NUM_HIDDEN, OUTPUTS, activation)
    nn_model.load_state_dict(torch.load(os.path.join(path_to_folder, filename), weights_only=True, map_location=torch.device('cpu')))
    nn_model.eval()
    train_val_test_name = f"{filename}_train_validation_test.pkl"
    train_val_test_path = os.path.join(path_to_folder, train_val_test_name)
    with open(train_val_test_path, "rb") as f:
        train_val_test = pickle.load(f)
    return nn_model, train_val_test

In [34]:
def make_layers_str(layers):
    if type(layers) is list:
        layers_str = "_".join([str(l) for l in layers])
    else:
        layers_str = str(layers)
    return layers_str

In [35]:
path_to_folder = "../pr_best_models/"
def open_best_model(num_nodes_layers, model_type, activation):
    layers_str = make_layers_str(num_nodes_layers)
    if model_type == "ernn":
        model_type_conv = "rnn"
    else:
        model_type_conv = model_type
    file_name = f"PR_{model_type_conv}_{activation}_wmse_{layers_str}_best"
    with torch.no_grad():
        nn_model, train_val_test = load_model(path_to_folder, file_name, model_type, activation, num_nodes_layers)
    return nn_model, train_val_test

In [36]:
with torch.no_grad():
    for model_type in ("ffnn", "ernn"):
        for activation in ("tanh", "relu"):
            for num_nodes_layers in num_nodes[model_type]:
                nn_model, train_val_test = open_best_model(num_nodes_layers, model_type, activation)
                n_parameters = sum(p.numel() for p in nn_model.parameters() if p.requires_grad)
                print(model_type, activation, num_nodes_layers, n_parameters)
                Xs_train, Xs_val, Xs_test, ys_train_true, ys_val_true, ys_test_true = get_train_val_test_data(train_val_test, batches)
                nn_model.eval()
                y_pred_train = torch.flatten(nn_model(Xs_train) * (maxes_tensor - mins_tensor) + mins_tensor)
                y_pred_val = torch.flatten(nn_model(Xs_val) * (maxes_tensor - mins_tensor) + mins_tensor)
                y_pred_test = torch.flatten(nn_model(Xs_test) * (maxes_tensor - mins_tensor) + mins_tensor)
                R2 = sklearn.metrics.r2_score(ys_train_true, y_pred_train)
                Q2_val = sklearn.metrics.r2_score(ys_val_true, y_pred_val)
                Q2 = sklearn.metrics.r2_score(ys_test_true, y_pred_test)
                print(f"R2={R2}, Q2_val={Q2_val} Q2={Q2}")

ffnn tanh [13, 13] 330
R2=0.9923860476161995, Q2_val=0.9893453810240781 Q2=0.9230508301907476
ffnn tanh [15, 15] 410
R2=0.9912057594595257, Q2_val=0.9905355349132119 Q2=0.9161660313341421
ffnn tanh [13, 13, 13] 512
R2=0.99303481335265, Q2_val=0.9917544972841039 Q2=0.9122196504110878
ffnn relu [13, 13] 330
R2=0.9899888168885509, Q2_val=0.9884375987138165 Q2=0.926759104693103
ffnn relu [15, 15] 410
R2=0.9890003991139283, Q2_val=0.9868076845263065 Q2=0.9207684430525696
ffnn relu [13, 13, 13] 512
R2=0.9892614780477673, Q2_val=0.9862120792796449 Q2=0.919293646526673
ernn tanh 10 225
R2=0.9932963553274432, Q2_val=0.9900842347281502 Q2=0.935237418436577
ernn tanh 15 410
R2=0.9935550265792654, Q2_val=0.9927078546574349 Q2=0.9105908868927053
ernn tanh 19 594
R2=0.9938389541507882, Q2_val=0.9926426173146267 Q2=0.9321190772762812
ernn relu 10 225
R2=0.9900909446788799, Q2_val=0.988858339240535 Q2=0.919735672825523
ernn relu 15 410
R2=0.9934242361168076, Q2_val=0.9913032723045403 Q2=0.934303039607